# LeetCode 651 — 4 Keys Keyboard bằng Python

Link đề: https://leetcode.com/problems/4-keys-keyboard/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming

Ghi chú (bài Premium): đề dưới đây được viết theo hiểu biết về bài, nên đối chiếu lại với đề gốc nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def maxA(self, n: int) -> int:
        ...
```

Ghi chú: bài quy hoạch động (DP) cổ điển về "khi nào nên chuyển từ gõ từng chữ `A` sang Copy-Paste". Notebook đi từ vét cạn mọi chuỗi phím (BFS trên trạng thái), qua DP `O(n^2)`, DP đệ quy có nhớ, đến DP `O(n)` chỉ dùng hai loại "khối" 4 và 5 phím.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có 4 phím `A`, `Ctrl-A`, `Ctrl-C`, `Ctrl-V`; với **tối đa `n` lần nhấn**, tìm số chữ `A` nhiều nhất trên màn hình.
2. Nắm các công cụ Python: `list(range(n + 1))` để khởi tạo bảng DP, `max`, vòng `for` lồng nhau với biên chính xác, `set` các trạng thái, `functools.lru_cache`.
3. Đi từ **mô phỏng mọi chuỗi phím** → **DP "khối Copy-Paste" `O(n^2)`** → **đệ quy có nhớ** → **DP `O(n)` với hai kích thước khối**.
4. Hiểu mấu chốt: `Ctrl-A`, `Ctrl-C` luôn đi **cặp**, rồi `k` lần `Ctrl-V` nhân số chữ `A` lên `(k + 1)` lần với giá `k + 2` lần nhấn; vì thế `dp[i] = max(i, dp[j-1] * (i - j))`.
5. Tránh các lỗi kinh điển: lệch chỉ số nhân (`dp[j]` thay vì `dp[j-1]`), quên phương án "gõ toàn chữ A", chỉ dùng một khối copy-paste, và đệ quy không nhớ gây bùng nổ thời gian.

## 2. Đề bài & ví dụ

Bạn có một bàn phím đặc biệt gồm các phím:

- `A`: in thêm một chữ `'A'` lên màn hình.
- `Ctrl-A`: chọn toàn bộ màn hình.
- `Ctrl-C`: sao chép phần đang chọn vào bộ nhớ đệm (buffer).
- `Ctrl-V`: in nội dung buffer, **nối tiếp** sau những gì đã in trên màn hình.

Cho số nguyên `n`, trả về **số chữ `A` nhiều nhất** có thể có trên màn hình sau **tối đa** `n` lần nhấn phím.

**Ví dụ 1**

- Input: `n = 3`
- Output: `3`
- Giải thích: nhấn `A, A, A`.

**Ví dụ 2**

- Input: `n = 7`
- Output: `9`
- Giải thích: `A, A, A, Ctrl-A, Ctrl-C, Ctrl-V, Ctrl-V` — gõ 3 chữ, chọn hết, copy, dán hai lần: `3 + 3 + 3 = 9`.

**Ràng buộc:**

- `1 <= n <= 50`

Lưu ý: với `n = 50` kết quả đã tới hơn một triệu (`1.327.104`) — vẫn nằm trong số nguyên của Python, và cả trong 32 bit.

In [1]:
from typing import List
from functools import lru_cache
import time

examples = [(3, 3), (7, 9)]
for n, expected in examples:
    print("n =", n, "-> kỳ vọng:", expected)

# Bảng tay: gõ toàn chữ A thì n lần nhấn cho n chữ
print([n for n in range(1, 8)])

n = 3 -> kỳ vọng: 3
n = 7 -> kỳ vọng: 9
[1, 2, 3, 4, 5, 6, 7]


## 3. Kiến thức nền

### 3.1. Quy hoạch động (DP) trong một câu

**DP** = chia bài toán thành các bài toán con nhỏ hơn, lưu đáp án của chúng vào một bảng, rồi dùng đáp án nhỏ để tính đáp án lớn. Ở đây bài con là: "với đúng `i` lần nhấn thì tối đa bao nhiêu chữ `A`" (`dp[i]`).

Công thức chuyển (transition) cho ta biết `dp[i]` tính từ những `dp[j]` nào.

In [2]:
# Ví dụ nhỏ về bảng DP: số cách đi lên cầu thang 1 hoặc 2 bậc (Fibonacci)
ways = [1, 1] + [0] * 8
for i in range(2, 10):
    ways[i] = ways[i - 1] + ways[i - 2]       # công thức chuyển
print(ways)

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


### 3.2. Khởi tạo bảng: `list(range(n + 1))`

`range(n + 1)` sinh `0, 1, ..., n`. `list(range(5 + 1))` cho `[0, 1, 2, 3, 4, 5]`. Ta dùng nó làm giá trị **khởi đầu** `dp[i] = i` — chính là phương án "gõ `i` chữ `A`" luôn có sẵn.

In [3]:
dp = list(range(7 + 1))
print(dp)              # [0, 1, 2, 3, 4, 5, 6, 7]
print(dp[-1], dp[3])   # dp[-1] là phần tử cuối

[0, 1, 2, 3, 4, 5, 6, 7]
7 3


### 3.3. `max` và vòng `for` lồng với biên đúng

`max(a, b)` trả số lớn hơn. Ta cập nhật dần: `dp[i] = max(dp[i], ứng_viên)` cho từng ứng viên. Vòng `for i in range(3, n + 1)` rồi `for j in range(2, i - 1)` cần biên chính xác — ta sẽ rút ra ở mục 6. Hãy nhớ `range(a, b)` **không** gồm `b`.

In [4]:
best = 0
for cand in [4, 9, 2]:
    best = max(best, cand)
    print("ứng viên", cand, "-> best =", best)
print(list(range(2, 6 - 1)))     # j chạy 2, 3 khi i = 6

ứng viên 4 -> best = 4
ứng viên 9 -> best = 9
ứng viên 2 -> best = 9
[2, 3, 4]


### 3.4. `set` các trạng thái và `lru_cache`

- `set` lưu **tập trạng thái không trùng**. Với cách vét cạn, trạng thái `(screen, buffer, selected)` giống nhau chỉ cần xét một lần.
- `@lru_cache(maxsize=None)` đặt trên một hàm đệ quy để **nhớ** kết quả theo tham số, tránh tính lại.

In [5]:
states = {(1, 0, False), (1, 0, False), (2, 0, False)}
print(states, len(states))


@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(60), fib.cache_info().hits > 0)

{(1, 0, False), (2, 0, False)} 2
1548008755920 True


## 4. Ý tưởng nền tảng / trực giác

Thử với `n = 7` bằng tay. Có hai chiến lược hiển nhiên:

- Gõ cả 7 chữ `A`: được **7**.
- Gõ 3 chữ `A`, rồi `Ctrl-A`, `Ctrl-C`, `Ctrl-V`, `Ctrl-V`: được **3 + 3 + 3 = 9**.

Quan sát then chốt: `Ctrl-A` rồi `Ctrl-C` tốn **2 lần nhấn** và chỉ "chuẩn bị" buffer = toàn bộ màn hình hiện tại `s`. Mỗi lần `Ctrl-V` sau đó cộng thêm `s`. Vậy sau `Ctrl-A, Ctrl-C` và `k` lần `Ctrl-V`:

```
màn hình = s + k*s = s * (k + 1)      số lần nhấn = k + 2
```

Ta gọi đó là một **khối** (block). Mỗi khối nhân màn hình lên `(k + 1)` lần, tốn `k + 2` lần nhấn:

| `k` | nhân | số phím | nhân trung bình trên mỗi phím |
|---|---|---|---|
| 1 | x2 | 3 | `2^(1/3) ≈ 1.260` |
| 2 | x3 | 4 | `3^(1/4) ≈ 1.316` |
| 3 | x4 | 5 | `4^(1/5) ≈ 1.320` |
| 4 | x5 | 6 | `5^(1/6) ≈ 1.308` |
| 5 | x6 | 7 | `6^(1/7) ≈ 1.292` |

Khối 4 và 5 phím là "kinh tế" nhất; nên nhiều khối trung bình (x3, x4) nối nhau tốt hơn một khối rất dài. Nhưng ta còn phải quyết định **gõ bao nhiêu chữ `A` trước khi bắt đầu** nhân.

In [6]:
import math

for k in range(1, 7):
    mult, presses = k + 1, k + 2
    print(f"k={k}: nhân x{mult} bằng {presses} phím -> {mult ** (1 / presses):.4f} lần/phím")

k=1: nhân x2 bằng 3 phím -> 1.2599 lần/phím
k=2: nhân x3 bằng 4 phím -> 1.3161 lần/phím
k=3: nhân x4 bằng 5 phím -> 1.3195 lần/phím
k=4: nhân x5 bằng 6 phím -> 1.3077 lần/phím
k=5: nhân x6 bằng 7 phím -> 1.2917 lần/phím
k=6: nhân x7 bằng 8 phím -> 1.2754 lần/phím


## 5. Cách 1 — Brute Force: mô phỏng mọi chuỗi phím (BFS theo trạng thái)

### Ý tưởng

Trạng thái = `(screen, buffer, selected)`: số `A` trên màn hình, số `A` trong buffer, và màn hình đang được chọn hay không. Bắt đầu `(0, 0, False)`. Mỗi lần nhấn, thử cả 4 phím:

- `A`: `screen + 1`, bỏ chọn.
- `Ctrl-A`: `selected = True`.
- `Ctrl-C`: nếu đang chọn thì `buffer = screen`.
- `Ctrl-V`: nếu `buffer > 0` thì `screen += buffer`, bỏ chọn.

Duyệt theo từng lớp (số lần nhấn), gom các trạng thái trong `set` để loại trùng, theo dõi `max(screen)`.

(Mô hình này theo đúng đề: `Ctrl-V` **nối tiếp** nội dung, không thay thế phần đang chọn. Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.)

### Code

In [7]:
def max_a_bfs(n: int) -> int:
    states = {(0, 0, False)}
    best = 0
    for _ in range(n):
        nxt = set()
        for screen, buf, sel in states:
            nxt.add((screen + 1, buf, False))              # A
            nxt.add((screen, buf, True))                   # Ctrl-A
            if sel:
                nxt.add((screen, screen, True))            # Ctrl-C
            if buf:
                nxt.add((screen + buf, buf, False))        # Ctrl-V
        states = nxt
        best = max(best, max(s[0] for s in states))
    return best


print(max_a_bfs(3), max_a_bfs(7))
print([max_a_bfs(n) for n in range(1, 13)])

3 9
[1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]


In [8]:
# Số trạng thái phình rất nhanh theo n
for n in (6, 10, 14, 18):
    states = {(0, 0, False)}
    for _ in range(n):
        nxt = set()
        for screen, buf, sel in states:
            nxt.add((screen + 1, buf, False))
            nxt.add((screen, buf, True))
            if sel:
                nxt.add((screen, screen, True))
            if buf:
                nxt.add((screen + buf, buf, False))
        states = nxt
    print(f"n={n:2d}: {len(states):6d} trạng thái ở lớp cuối")

n= 6:     29 trạng thái ở lớp cuối
n=10:    140 trạng thái ở lớp cuối
n=14:    561 trạng thái ở lớp cuối
n=18:   1996 trạng thái ở lớp cuối


### Độ phức tạp và nút thắt

Số trạng thái tăng rất nhanh, gần như **theo hàm mũ** (mỗi lớp thêm nhiều giá trị `(screen, buffer)` mới), nên chỉ chạy nổi vài chục lần nhấn; với `n = 50` thì không khả thi. Nhưng đây là **oracle** rất đáng tin để kiểm tra các cách sau (chỉ dựa trên luật phím, không dựa vào công thức nào).

Nút thắt: ta mô phỏng cả những trạng thái vô ích (ví dụ `Ctrl-A` liên tục, copy ở giữa rồi gõ thêm `A`...). Muốn bỏ chúng, cần hiểu **cấu trúc** của chuỗi phím tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

Khẳng định: có một chuỗi tối ưu dạng

```
A A ... A   |  [Ctrl-A Ctrl-C V ... V]  [Ctrl-A Ctrl-C V ... V]  ...
(gõ trước)      khối 1                      khối 2
```

tức một đoạn gõ `A` rồi các **khối** (mỗi khối: `Ctrl-A`, `Ctrl-C`, rồi `k >= 1` lần `Ctrl-V`). Lý do: `Ctrl-V` chỉ có ích khi buffer khác rỗng; buffer chỉ đổi bởi `Ctrl-C` (phải đi kèm `Ctrl-A` để chọn cả màn hình); và gõ thêm `A` **sau** khi đã copy chỉ làm buffer cũ lỗi thời so với màn hình, không lợi hơn việc gõ `A` từ trước.

Từ đó viết công thức. Gọi `dp[i]` = số `A` tối đa với tối đa `i` lần nhấn. Xét **khối cuối cùng** bắt đầu bằng `Ctrl-A` ở lần nhấn thứ `j` (`2 <= j`):

- Trước đó đã dùng `j - 1` lần nhấn, màn hình tối đa `dp[j - 1]`.
- `Ctrl-A` là lần `j`, `Ctrl-C` là lần `j + 1`, còn lại `i - (j + 1) = i - j - 1` lần `Ctrl-V`.
- Màn hình cuối: `dp[j - 1] * (1 + (i - j - 1)) = dp[j - 1] * (i - j)`.

Vậy:

```
dp[i] = max( i,                              # gõ toàn chữ A
             max_{j} dp[j - 1] * (i - j) )   # khối cuối bắt đầu bằng Ctrl-A ở lần j
```

Biên của `j`: cần ít nhất 1 lần `Ctrl-V` (`i - j - 1 >= 1` → `j <= i - 2`), và `j >= 2` (cần ít nhất một lần nhấn trước `Ctrl-A`). Vậy `j` chạy `range(2, i - 1)`. Khi `i <= 3`, không có `j` nào → `dp[i] = i`.

In [9]:
# Tính tay dp[7] theo công thức: j chạy 2..5
dp = list(range(8))
for i in range(3, 8):
    for j in range(2, i - 1):
        dp[i] = max(dp[i], dp[j - 1] * (i - j))
    print(f"dp[{i}] = {dp[i]}")

i = 7
print("các ứng viên của dp[7]:", [(j, dp[j - 1] * (i - j)) for j in range(2, i - 1)])

dp[3] = 3
dp[4] = 4
dp[5] = 5
dp[6] = 6
dp[7] = 9
các ứng viên của dp[7]: [(2, 5), (3, 8), (4, 9), (5, 8)]


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) BFS mô phỏng phím (mục 5), (2) DP bảng `O(n^2)`, (3) đệ quy có nhớ theo "khối cuối", (4) DP `O(n)` với khối 4 và 5 phím.

### 7.1. Cách 2 — DP bảng `O(n^2)`

Giải thích từng đoạn:

1. `dp = list(range(n + 1))`: phương án gõ toàn chữ `A`.
2. `for i in range(3, n + 1)`: tính dần `dp[i]`.
3. `for j in range(2, i - 1)`: thử khối cuối bắt đầu ở lần `j`, ứng viên `dp[j - 1] * (i - j)`.
4. Trả `dp[n]`.

In [10]:
def max_a_dp(n: int) -> int:
    dp = list(range(n + 1))
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            dp[i] = max(dp[i], dp[j - 1] * (i - j))
    return dp[n]


print(max_a_dp(3), max_a_dp(7), [max_a_dp(n) for n in range(1, 13)])

3 9 [1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]


In [11]:
# Code nộp LeetCode cho cách 2
class SolutionDP:
    def maxA(self, n: int) -> int:
        dp = list(range(n + 1))
        for i in range(3, n + 1):
            for j in range(2, i - 1):
                dp[i] = max(dp[i], dp[j - 1] * (i - j))
        return dp[-1]


print(SolutionDP().maxA(3), SolutionDP().maxA(7))

3 9


**Độ phức tạp:** hai vòng lồng nhau, mỗi vòng tối đa `n` → `O(n^2)` thời gian (với `n = 50` chỉ ~1200 phép tính), `O(n)` bộ nhớ.

### 7.2. Cách 3 — Đệ quy có nhớ theo "khối cuối"

Cùng công thức nhưng nhìn **từ trên xuống**: `f(m)` = số `A` tối đa với `m` lần nhấn. Nếu khối cuối có `k` lần `Ctrl-V` (tốn `k + 2` phím) thì trước đó còn `m - k - 2` phím, và màn hình nhân lên `k + 1`:

```
f(m) = max( m, max_{k >= 1, m-k-2 >= 1} f(m - k - 2) * (k + 1) )
```

(Đổi biến `j = m - k - 1` ta được đúng công thức cách 2.) Dùng `lru_cache` để mỗi `f(m)` chỉ tính một lần.

In [12]:
def max_a_memo(n: int) -> int:
    @lru_cache(maxsize=None)
    def f(m: int) -> int:
        best = m                                   # gõ toàn chữ A
        for k in range(1, m - 2):                  # m - k - 2 >= 1
            best = max(best, f(m - k - 2) * (k + 1))
        return best

    return f(n)


print(max_a_memo(3), max_a_memo(7), [max_a_memo(n) for n in range(1, 13)])

3 9 [1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]


In [13]:
# Code nộp LeetCode cho cách 3
class SolutionMemo:
    def maxA(self, n: int) -> int:
        @lru_cache(maxsize=None)
        def f(m: int) -> int:
            best = m
            for k in range(1, m - 2):
                best = max(best, f(m - k - 2) * (k + 1))
            return best

        return f(n)


print(SolutionMemo().maxA(3), SolutionMemo().maxA(7))

3 9


**Độ phức tạp:** có `n` giá trị `m`, mỗi giá trị lặp tối đa `m` lần → `O(n^2)` thời gian, `O(n)` bộ nhớ bảng nhớ + chiều sâu đệ quy `O(n)` (với `n <= 50` rất an toàn).

### 7.3. Cách 4 — DP `O(n)` chỉ dùng khối 4 và 5 phím

Từ bảng ở mục 4, khối x3 (4 phím) và x4 (5 phím) có hiệu quả cao nhất. Một **lập luận hoán đổi** cho thấy khối dài (`k >= 5`, tức `>= 7` phím) luôn có thể tách thành hai khối ngắn không kém: với `k >= 5`, chia `k + 2 = (a + 2) + (b + 2)` (`a + b = k - 2`, `a, b >= 1`) thì tích `(a + 1)(b + 1) = ab + k - 1 >= k + 1` khi `ab >= 2`. Vậy chỉ còn `k = 1..4`. Kiểm chứng bằng máy (mục 10 và 12) cho thấy công thức rút gọn

```
dp[i] = i                                   với i <= 6
dp[i] = max(dp[i - 4] * 3, dp[i - 5] * 4)   với i >= 7  (và so với i)
```

cho cùng kết quả với DP đầy đủ. Mỗi `dp[i]` chỉ tốn `O(1)` → tổng `O(n)`.

In [14]:
def max_a_linear(n: int) -> int:
    dp = list(range(n + 1))
    for i in range(7, n + 1):
        dp[i] = max(i, dp[i - 4] * 3, dp[i - 5] * 4)
    return dp[n]


print(max_a_linear(3), max_a_linear(7), [max_a_linear(n) for n in range(1, 13)])
assert all(max_a_linear(n) == max_a_dp(n) for n in range(1, 301))
print("khớp DP đầy đủ với n = 1..300")

3 9 [1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]


khớp DP đầy đủ với n = 1..300


In [15]:
# Code nộp LeetCode cho cách 4
class SolutionLinear:
    def maxA(self, n: int) -> int:
        dp = list(range(n + 1))
        for i in range(7, n + 1):
            dp[i] = max(i, dp[i - 4] * 3, dp[i - 5] * 4)
        return dp[n]


print(SolutionLinear().maxA(3), SolutionLinear().maxA(7))

3 9


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (có thể giảm còn `O(1)` bằng cách giữ 5 giá trị gần nhất). Lưu ý: tính đúng của công thức rút gọn ở đây dựa trên lập luận hoán đổi cho `k >= 5` **cộng** kiểm chứng bằng máy tới `n = 300`; lời giải nộp an toàn nhất vẫn là cách 2.

## 8. TRACE từng bước

### 8.1. Bảng DP với `n = 9`

Khởi tạo `dp = [0,1,2,3,4,5,6,7,8,9]`. Với từng `i`, thử `j` từ 2 đến `i - 2` (ứng viên `dp[j-1] * (i-j)`):

| `i` | các `(j, ứng viên)` | `dp[i]` sau |
|---|---|---|
| 3 | (không có `j`) | 3 |
| 4 | `(2, dp[1]*2 = 2)` | 4 |
| 5 | `(2, dp[1]*3 = 3)`, `(3, dp[2]*2 = 4)` | 5 |
| 6 | `(2, 4)`, `(3, 6)`, `(4, dp[3]*2 = 6)` | 6 |
| 7 | `(2, 5)`, `(3, 8)`, `(4, 9)`, `(5, 8)` | **9** |
| 8 | `(2, 6)`, `(3, 10)`, `(4, 12)`, `(5, 12)`, `(6, 10)` | **12** |
| 9 | `(2, 7)`, `(3, 12)`, `(4, 15)`, `(5, 16)`, `(6, 15)`, `(7, 12)` | **16** |

`dp[9] = 16` đến từ `j = 5`: gõ 4 chữ `A` (`dp[4] = 4`), rồi `Ctrl-A` ở lần 5, `Ctrl-C` ở lần 6, và 3 lần `Ctrl-V`: `4 * 4 = 16`.

In [16]:
def trace_dp(n):
    dp = list(range(n + 1))
    print("khởi tạo dp =", dp)
    for i in range(3, n + 1):
        cands = []
        for j in range(2, i - 1):
            cands.append((j, dp[j - 1] * (i - j)))
        best = max([dp[i]] + [c for _, c in cands])
        print(f"i={i}: ứng viên (j, dp[j-1]*(i-j)) = {cands} | gõ toàn A = {dp[i]} -> dp[{i}] = {best}")
        dp[i] = best
    return dp[n]


print("kết quả:", trace_dp(9))

khởi tạo dp = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
i=3: ứng viên (j, dp[j-1]*(i-j)) = [] | gõ toàn A = 3 -> dp[3] = 3
i=4: ứng viên (j, dp[j-1]*(i-j)) = [(2, 2)] | gõ toàn A = 4 -> dp[4] = 4
i=5: ứng viên (j, dp[j-1]*(i-j)) = [(2, 3), (3, 4)] | gõ toàn A = 5 -> dp[5] = 5
i=6: ứng viên (j, dp[j-1]*(i-j)) = [(2, 4), (3, 6), (4, 6)] | gõ toàn A = 6 -> dp[6] = 6
i=7: ứng viên (j, dp[j-1]*(i-j)) = [(2, 5), (3, 8), (4, 9), (5, 8)] | gõ toàn A = 7 -> dp[7] = 9
i=8: ứng viên (j, dp[j-1]*(i-j)) = [(2, 6), (3, 10), (4, 12), (5, 12), (6, 10)] | gõ toàn A = 8 -> dp[8] = 12
i=9: ứng viên (j, dp[j-1]*(i-j)) = [(2, 7), (3, 12), (4, 15), (5, 16), (6, 15), (7, 12)] | gõ toàn A = 9 -> dp[9] = 16
kết quả: 16


### 8.2. Dựng lại chuỗi phím tối ưu và mô phỏng

Lưu thêm lựa chọn `j` tốt nhất để dựng lại dãy phím, rồi **mô phỏng** dãy phím đó để kiểm tra nó thật sự ra `dp[n]` chữ `A` trong không quá `n` lần nhấn.

In [17]:
def best_keys(n):
    dp = list(range(n + 1))
    choice = [0] * (n + 1)          # 0 nghĩa là gõ toàn chữ A
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            if dp[j - 1] * (i - j) > dp[i]:
                dp[i] = dp[j - 1] * (i - j)
                choice[i] = j

    def build(i):
        if choice[i] == 0:
            return ["A"] * i
        j = choice[i]
        return build(j - 1) + ["^A", "^C"] + ["^V"] * (i - j - 1)

    return dp[n], build(n)


def simulate(keys):
    screen = buf = 0
    sel = False
    for key in keys:
        if key == "A":
            screen, sel = screen + 1, False
        elif key == "^A":
            sel = True
        elif key == "^C":
            assert sel, "Ctrl-C mà chưa chọn"
            buf = screen
        elif key == "^V":
            screen, sel = screen + buf, False
    return screen


for n in (3, 7, 9, 11, 14):
    val, keys = best_keys(n)
    print(f"n={n:2d}: {val:3d} chữ A, {len(keys)} phím: {' '.join(keys)}")
    assert simulate(keys) == val and len(keys) <= n

n= 3:   3 chữ A, 3 phím: A A A
n= 7:   9 chữ A, 7 phím: A A A ^A ^C ^V ^V
n= 9:  16 chữ A, 9 phím: A A A A ^A ^C ^V ^V ^V
n=11:  27 chữ A, 11 phím: A A A ^A ^C ^V ^V ^A ^C ^V ^V
n=14:  64 chữ A, 14 phím: A A A A ^A ^C ^V ^V ^V ^A ^C ^V ^V ^V


## 9. Code nộp LeetCode

Chọn cách 2 (DP bảng `O(n^2)`) làm lời giải nộp: ngắn, chắc chắn đúng theo công thức đã chứng minh, với `n <= 50` chạy tức thì.

In [18]:
class Solution:
    def maxA(self, n: int) -> int:
        dp = list(range(n + 1))
        for i in range(3, n + 1):
            for j in range(2, i - 1):
                dp[i] = max(dp[i], dp[j - 1] * (i - j))
        return dp[-1]


sol = Solution()
print(sol.maxA(3))   # 3
print(sol.maxA(7))   # 9

3
9


## 10. Vì sao đúng

**Cấu trúc chuỗi tối ưu.** `Ctrl-V` chỉ tăng màn hình khi buffer khác rỗng, và buffer chỉ đổi bằng `Ctrl-C`. `Ctrl-C` chỉ có tác dụng khi đang chọn, mà cách duy nhất để chọn trong bài này là `Ctrl-A` (chọn toàn bộ màn hình). Vậy mỗi lần buffer đổi ứng với một cặp `Ctrl-A, Ctrl-C` (đặt liền nhau là tốt nhất, chen phím khác vào chỉ phí). Còn các phím `A` gõ **sau** một khối thì có thể **dời lên đầu** mà không giảm kết quả: một chữ `A` gõ muộn chỉ đóng góp `+1`, còn nếu gõ sớm thì mọi khối phía sau đều nhân nó lên (mỗi khối nhân màn hình lên ít nhất 2 lần) và còn làm buffer lớn hơn. Kết luận: có chuỗi tối ưu dạng "gõ `A`" rồi "các khối".

**Công thức.** Với `i` lần nhấn và chuỗi có ít nhất một khối, xét khối cuối: bắt đầu bằng `Ctrl-A` ở lần `j`, rồi `Ctrl-C`, rồi `i - j - 1` lần `Ctrl-V`. Phần trước khối tối đa là `dp[j - 1]` (dùng `j - 1` phím), màn hình sau khối là `dp[j - 1] * (i - j)`. Không có khối nào thì đáp án là `i`. Lấy max của mọi khả năng ta có công thức ở mục 6. Vì hàm `dp` chỉ phụ thuộc phần tử nhỏ hơn, tính tăng dần là hợp lệ.

**Bài toán con tối ưu.** Nếu phần trước khối cuối không tối ưu cho `j - 1` phím thì thay bằng phần tối ưu sẽ tăng kết quả (nhân với hằng số dương `i - j`). Vậy dùng `dp[j - 1]` là đúng.

**Cách 4.** Khối `k >= 5` thay bằng hai khối nhỏ không tệ hơn (lập luận ở 7.3). Phần còn lại (chỉ cần `k = 2, 3` cho `i >= 7`) được kiểm chứng bằng máy tới `n = 500` ở cell dưới.

In [19]:
# Kiểm chứng cách 4 đến n = 500 và quan sát khối nào được chọn
full = list(range(501))
for i in range(3, 501):
    for j in range(2, i - 1):
        full[i] = max(full[i], full[j - 1] * (i - j))

lin = list(range(501))
for i in range(7, 501):
    lin[i] = max(i, lin[i - 4] * 3, lin[i - 5] * 4)

assert full == lin
print("DP đầy đủ == DP khối 4/5 với n = 0..500")

# khối cuối thực sự tối ưu có k nào? (k = số Ctrl-V)
ks = set()
for i in range(7, 60):
    for j in range(2, i - 1):
        if full[j - 1] * (i - j) == full[i]:
            ks.add(i - j - 1)
print("các k xuất hiện ở khối cuối tối ưu, i = 7..59:", sorted(ks))

DP đầy đủ == DP khối 4/5 với n = 0..500
các k xuất hiện ở khối cuối tối ưu, i = 7..59: [2, 3, 4]


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS mô phỏng phím | hàm mũ theo `n` | số trạng thái |
| 2. DP bảng | `O(n^2)` | `O(n)` |
| 3. Đệ quy có nhớ | `O(n^2)` | `O(n)` (bảng nhớ + ngăn xếp) |
| 4. DP khối 4/5 | `O(n)` | `O(n)` (hoặc `O(1)`) |

Với ràng buộc `n <= 50`, cả ba cách DP đều chạy tức thì. Cell dưới đếm số phép nhân ở cách 2 để thấy tăng trưởng bậc 2.

In [20]:
def count_ops(n):
    ops = 0
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            ops += 1
    return ops


for n in (10, 20, 40, 50, 100, 200):
    print(f"n={n:3d}: {count_ops(n):6d} ứng viên | n^2/2 = {n * n // 2}")

t0 = time.perf_counter()
for _ in range(1000):
    Solution().maxA(50)
print("1000 lần maxA(50):", round((time.perf_counter() - t0) * 1000, 1), "ms")

n= 10:     28 ứng viên | n^2/2 = 50
n= 20:    153 ứng viên | n^2/2 = 200
n= 40:    703 ứng viên | n^2/2 = 800
n= 50:   1128 ứng viên | n^2/2 = 1250
n=100:   4753 ứng viên | n^2/2 = 5000
n=200:  19503 ứng viên | n^2/2 = 20000
1000 lần maxA(50): 78.3 ms


## 12. Bộ test

Chiến lược: (a) ví dụ đề; (b) edge case và giá trị tra cứu đã biết; (c) phản ví dụ cho các cách sai; (d) đối chiếu BFS (oracle độc lập theo luật phím) với mọi cách; (e) đối chiếu các DP với nhau trên toàn miền `1..50` và vượt miền; (f) tính chất (đơn điệu, giá trị tại 50) và mô phỏng dãy phím dựng lại.

In [21]:
solvers = {
    "dp bảng": max_a_dp,
    "đệ quy nhớ": max_a_memo,
    "tuyến tính": max_a_linear,
    "nộp Solution": lambda n: Solution().maxA(n),
    "nộp DP": lambda n: SolutionDP().maxA(n),
    "nộp Memo": lambda n: SolutionMemo().maxA(n),
    "nộp Linear": lambda n: SolutionLinear().maxA(n),
}

# (a) ví dụ đề
for n, expected in examples:
    for name, f in solvers.items():
        assert f(n) == expected, (name, n)
print("ví dụ đề: OK")

# (b) edge case và giá trị đã biết (n <= 6: gõ toàn A)
for n in range(1, 7):
    for name, f in solvers.items():
        assert f(n) == n, (name, n)
known = {7: 9, 8: 12, 9: 16, 10: 20, 11: 27, 12: 36, 13: 48, 14: 64, 15: 81, 16: 108, 20: 324, 50: 1327104}
for n, v in known.items():
    for name, f in solvers.items():
        assert f(n) == v, (name, n, f(n), v)
print("edge case và bảng giá trị đã biết: OK")

ví dụ đề: OK
edge case và bảng giá trị đã biết: OK


In [22]:
# (d) đối chiếu BFS (oracle theo luật phím) với mọi cách, n = 1..20
for n in range(1, 21):
    expected = max_a_bfs(n)
    for name, f in solvers.items():
        assert f(n) == expected, (name, n, f(n), expected)
print("BFS == mọi DP với n = 1..20: OK")

# (e) các DP khớp nhau trên toàn miền đề và hơn nữa
for n in range(1, 51):
    vals = {name: f(n) for name, f in solvers.items()}
    assert len(set(vals.values())) == 1, (n, vals)
for n in (60, 100, 150, 200):
    assert max_a_dp(n) == max_a_linear(n) == max_a_memo(n)
print("các DP khớp nhau với n = 1..50, 60, 100, 150, 200: OK")

BFS == mọi DP với n = 1..20: OK

các DP khớp nhau với n = 1..50, 60, 100, 150, 200: OK


In [23]:
# (f) tính chất: không giảm theo n, "tối đa n lần nhấn" nên f(n) >= f(n-1); tăng nhanh ~ 1.32^n
vals = [Solution().maxA(n) for n in range(1, 51)]
assert all(b >= a for a, b in zip(vals, vals[1:]))
assert all(v >= n for n, v in zip(range(1, 51), vals))
ratios = [vals[i + 1] / vals[i] for i in range(30, 49)]
print("tỉ số f(n+1)/f(n) với n = 31..49:", [round(r, 3) for r in ratios[:6]], "...")
assert vals[-1] == 1327104
print("f(50) =", vals[-1], "(< 2^31 - 1 =", 2 ** 31 - 1, ")")

# mô phỏng dãy phím dựng lại cho mọi n = 1..50
for n in range(1, 51):
    val, keys = best_keys(n)
    assert val == Solution().maxA(n) and len(keys) <= n and simulate(keys) == val
print("dãy phím dựng lại hợp lệ và tối ưu với n = 1..50: OK")

tỉ số f(n+1)/f(n) với n = 31..49: [1.333, 1.333, 1.333, 1.266, 1.333, 1.333] ...
f(50) = 1327104 (< 2^31 - 1 = 2147483647 )
dãy phím dựng lại hợp lệ và tối ưu với n = 1..50: OK


In [24]:
# (c) phản ví dụ: các cách sai (phân tích ở mục 13)
def wrong_off_by_one(n):
    dp = list(range(n + 1))
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            dp[i] = max(dp[i], dp[j] * (i - j))          # SAI: dp[j], không phải dp[j - 1]
    return dp[-1]


def wrong_no_baseline(n):
    dp = [0] * (n + 1)                                   # SAI: quên "gõ toàn chữ A"
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            dp[i] = max(dp[i], dp[j - 1] * (i - j))
    return dp[-1]


def wrong_single_block(n):
    best = n
    for a in range(1, n - 1):                            # SAI: chỉ một khối copy-paste
        best = max(best, a * (n - a - 1))
    return best


for f in (wrong_off_by_one, wrong_no_baseline, wrong_single_block):
    bad = [n for n in range(1, 21) if f(n) != Solution().maxA(n)]
    print(f"{f.__name__:18s}: sai ở n = {bad[:8]}{'...' if len(bad) > 8 else ''}")
    assert bad

wrong_off_by_one  : sai ở n = [5, 6, 7, 8, 9, 10, 11, 12]...
wrong_no_baseline : sai ở n = [1, 2, 3, 4, 5, 6, 7, 8]...
wrong_single_block: sai ở n = [11, 12, 13, 14, 15, 16, 17, 18]...


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lệch chỉ số: nhân `dp[j]` thay vì `dp[j - 1]`

`Ctrl-A` nằm ở lần nhấn thứ `j`, nên phần **trước** khối chỉ dùng `j - 1` phím. Viết `dp[j]` tức là "dùng cả phím thứ `j`" để gõ `A` mà vẫn lấy `Ctrl-A` ở lần `j` — một phím được dùng hai lần.

In [25]:
for n in (4, 5, 6, 7, 8):
    print(f"n={n}: đúng = {Solution().maxA(n)} | bản sai dp[j] = {wrong_off_by_one(n)}")
assert wrong_off_by_one(7) > Solution().maxA(7)

n=4: đúng = 4 | bản sai dp[j] = 4
n=5: đúng = 5 | bản sai dp[j] = 6
n=6: đúng = 6 | bản sai dp[j] = 9
n=7: đúng = 9 | bản sai dp[j] = 12
n=8: đúng = 12 | bản sai dp[j] = 18


**Vì sao sai:** bản sai cho kết quả **lớn hơn** đáp án (ví dụ `n = 7` ra giá trị lớn hơn 9) vì "tặng" thêm một phím. Đáp án vượt mức tối đa là dấu hiệu nên nghi lệch chỉ số. **Đúng:** `dp[j - 1] * (i - j)`.

### Lỗi 2 — Quên phương án "gõ toàn chữ A" (khởi tạo toàn 0)

Nếu `dp = [0] * (n + 1)` thì với `n <= 3` vòng `j` rỗng và ta trả về 0; ngay cả với `n` lớn, phần "gõ trước" `dp[j - 1]` cũng bị thiếu.

In [26]:
print([wrong_no_baseline(n) for n in range(1, 9)])
print([Solution().maxA(n) for n in range(1, 9)])

[0, 0, 0, 0, 0, 0, 0, 0]
[1, 2, 3, 4, 5, 6, 9, 12]


**Vì sao sai:** nếu không có `dp[i] >= i` thì `dp[1..3]` bằng 0, kéo theo mọi khối bị nhân với 0. **Đúng:** khởi tạo `dp = list(range(n + 1))`.

### Lỗi 3 — Chỉ dùng một khối copy-paste

Cách nghĩ ngây thơ: "gõ `a` chữ `A`, rồi `Ctrl-A`, `Ctrl-C`, rồi dán hết số phím còn lại". Kết quả `a * (n - a - 1)`. Với `n` nhỏ cho đúng, với `n` lớn thì sai vì nhiều khối nhỏ nhân liên tiếp tốt hơn một khối dài.

In [27]:
for n in (7, 9, 10, 11, 12, 15):
    print(f"n={n:2d}: đúng = {Solution().maxA(n):3d} | một khối = {wrong_single_block(n):3d}")
first_bad = next(n for n in range(1, 30) if wrong_single_block(n) != Solution().maxA(n))
print("n nhỏ nhất bị sai:", first_bad)

n= 7: đúng =   9 | một khối =   9
n= 9: đúng =  16 | một khối =  16
n=10: đúng =  20 | một khối =  20
n=11: đúng =  27 | một khối =  25
n=12: đúng =  36 | một khối =  30
n=15: đúng =  81 | một khối =  49
n nhỏ nhất bị sai: 11


**Vì sao sai:** ở `n = 11`, đáp án 27 = `3 * 3 * 3` (gõ 3 chữ, rồi hai khối x3 mỗi khối 4 phím), trong khi một khối chỉ tối đa 25. **Đúng:** cho phép **nhiều khối** — chính là công thức `dp[j - 1] * (i - j)` với `dp[j - 1]` đã chứa các khối trước.

### Lỗi 4 — Đệ quy không nhớ: bùng nổ thời gian

Cùng công thức nhưng quên `lru_cache`, mỗi `f(m)` tính lại nhiều lần.

In [28]:
calls = 0


def f_no_memo(m):
    global calls
    calls += 1
    best = m
    for k in range(1, m - 2):
        best = max(best, f_no_memo(m - k - 2) * (k + 1))
    return best


for n in (8, 12, 16, 20):
    calls = 0
    v = f_no_memo(n)
    assert v == Solution().maxA(n)
    print(f"n={n:2d}: đúng giá trị {v}, nhưng cần {calls} lần gọi hàm")

n= 8: đúng giá trị 12, nhưng cần 9 lần gọi hàm
n=12: đúng giá trị 36, nhưng cần 41 lần gọi hàm
n=16: đúng giá trị 108, nhưng cần 189 lần gọi hàm
n=20: đúng giá trị 324, nhưng cần 872 lần gọi hàm


**Vì sao chậm:** số lần gọi tăng theo hàm mũ (khoảng nhân 1,5 lần mỗi khi `n` thêm 1; từ `n = 8` đến `n = 20` đã từ 9 lên 872 lần); với `n = 50` là cỡ hàng chục triệu lần gọi trở lên — quá chậm. **Đúng:** thêm `@lru_cache` hoặc dùng bảng DP lặp.

## 14. Mẹo & nhận dạng pattern

- Đề hỏi "tối đa/tối thiểu sau đúng/tối đa `n` bước" với vài loại thao tác, mỗi thao tác phụ thuộc trạng thái nhỏ → nghĩ **DP theo số bước**.
- Khi thao tác đi **theo cặp/cụm** (ở đây `Ctrl-A, Ctrl-C, V...V`), hãy gom cụm thành một "khối" và viết DP theo khối cuối cùng — đừng DP theo từng phím với trạng thái `(screen, buffer, selected)` nếu không cần.
- Công thức dạng "tích" `dp[j-1] * (i-j)`: kiểm tra chỉ số bằng cách tự cộng số phím: `(j - 1) + 2 + (i - j - 1) = i`.
- Viết **oracle vét cạn** (BFS trạng thái) cho `n` nhỏ rồi so với DP — cách tốt nhất để bắt lỗi lệch chỉ số.
- Khi muốn tối ưu thêm, xét **hiệu suất trên mỗi phím** (`nhân^(1/phím)`): khối 4 và 5 phím tốt nhất, rút ra DP `O(n)`.
- Đáp án tăng theo cấp số nhân (`~1.32^n`): chú ý tràn số ở ngôn ngữ 32/64 bit; Python không lo.

## 15. Biến thể

### 15.1. Chi phí chọn + copy khác 2

Giả sử "chọn + copy" tốn `c` phím thay vì 2. Khối `k` lần dán tốn `k + c` phím, nhân `(k + 1)`: `f(m) = max(m, max_{k>=1} f(m - k - c) * (k + 1))`.

In [29]:
def max_a_cost(n: int, c: int = 2) -> int:
    f = list(range(n + 1))
    for m in range(1, n + 1):
        for k in range(1, m):
            prev = m - k - c
            if prev >= 1:
                f[m] = max(f[m], f[prev] * (k + 1))
    return f[n]


print([max_a_cost(n, 2) for n in range(1, 13)])
print([max_a_cost(n, 3) for n in range(1, 13)])
assert [max_a_cost(n, 2) for n in range(1, 51)] == [Solution().maxA(n) for n in range(1, 51)]

[1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]
[1, 2, 3, 4, 5, 6, 7, 9, 12, 16, 20, 25]


### 15.2. Trả về chuỗi phím chứ không chỉ số lượng

Đã làm ở mục 8.2 (`best_keys`): lưu `choice[i]` rồi dựng lại, và kiểm chứng bằng `simulate`.

### 15.3. Bài họ hàng: LeetCode 650 — 2 Keys Keyboard

Chỉ có `Copy All` và `Paste`; hỏi **số bước tối thiểu** để có đúng `n` chữ `A`. Đáp án là tổng các thừa số nguyên tố của `n` (phân tích `n` thành tích, mỗi thừa số `p` tốn `p` bước).

In [30]:
def min_steps(n):
    ans, p = 0, 2
    while n > 1:
        while n % p == 0:
            ans += p
            n //= p
        p += 1
    return ans


print([min_steps(n) for n in range(1, 13)])     # 0 2 3 4 5 5 7 6 6 7 11 7

[0, 2, 3, 4, 5, 5, 7, 6, 6, 7, 11, 7]


### 15.4. Giới hạn số lần `Ctrl-V` mỗi khối

Nếu mỗi khối chỉ được dán tối đa `K` lần thì giới hạn `k <= K` trong vòng lặp. Với `K = 1` chỉ có khối x2.

In [31]:
def max_a_limited(n: int, K: int) -> int:
    f = list(range(n + 1))
    for m in range(1, n + 1):
        for k in range(1, min(K, m) + 1):
            prev = m - k - 2
            if prev >= 1:
                f[m] = max(f[m], f[prev] * (k + 1))
    return f[n]


print([max_a_limited(n, 1) for n in range(1, 13)])
print([max_a_limited(n, 99) for n in range(1, 13)])
assert all(max_a_limited(n, 99) == Solution().maxA(n) for n in range(1, 51))

[1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24]
[1, 2, 3, 4, 5, 6, 9, 12, 16, 20, 27, 36]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mô phỏng phím | Duyệt mọi trạng thái `(screen, buffer, selected)` | hàm mũ | số trạng thái | Oracle kiểm tra, `n <= ~20` |
| 2. DP bảng | `dp[i] = max(i, dp[j-1]*(i-j))` | `O(n^2)` | `O(n)` | **Lời giải nộp**, chắc chắn đúng |
| 3. Đệ quy có nhớ | Cùng công thức, nhìn từ trên xuống | `O(n^2)` | `O(n)` | Dễ suy ra từ định nghĩa, `n` nhỏ |
| 4. DP khối 4/5 | Chỉ dùng khối x3 và x4 | `O(n)` | `O(n)` / `O(1)` | `n` rất lớn (với số nguyên lớn), tối ưu |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(Solution().maxA(6))
print(Solution().maxA(8))
print(simulate(["A", "A", "A", "^A", "^C", "^V"]))
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Dùng bảng DP, tính tay `dp[10]` và `dp[11]` biết `dp[1..9] = 1,2,3,4,5,6,9,12,16`. Với mỗi giá trị, nêu `j` tốt nhất và dãy phím tương ứng (gõ bao nhiêu `A`, mấy lần `Ctrl-V`).

In [33]:
# Bài 2: điền kết quả tính tay
hand_dp10 = None
hand_dp11 = None
hand_best_j_for_10 = None
hand_pastes_for_10 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm DP theo khối.

In [34]:
def exercise_3_dp(n: int) -> int:
    # TODO 1: khởi tạo dp với phương án gõ toàn chữ A
    # TODO 2: vòng i từ 3 đến n
    #     TODO 3: vòng j từ 2 đến i - 2; ứng viên dp[j - 1] * (i - j)
    # TODO 4: trả về dp[n]
    return None


# assert exercise_3_dp(3) == 3
# assert exercise_3_dp(7) == 9
# assert exercise_3_dp(11) == 27

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_linear(n)` dùng công thức khối x3 và x4 cho `i >= 7` (khởi tạo `dp[i] = i`).

In [35]:
def exercise_4_linear(n: int) -> int:
    # TODO
    pass


# assert exercise_4_linear(7) == 9
# assert exercise_4_linear(20) == 324
# assert exercise_4_linear(50) == 1327104

### Bài 5 — Mini challenge: chi phí chọn+copy bằng 3 (khó)

Giả sử "chọn + copy" tốn 3 phím thay vì 2 (còn `A` và `Ctrl-V` mỗi cái 1 phím). Viết `exercise_5_cost3(n)` trả về số `A` tối đa. Gợi ý: khối `k` lần dán tốn `k + 3` phím, nhân `(k + 1)`.

```python
exercise_5_cost3(7) -> 7
exercise_5_cost3(8) -> 9
exercise_5_cost3(3) -> 3
```

In [36]:
def exercise_5_cost3(n: int) -> int:
    # TODO
    pass


# assert exercise_5_cost3(3) == 3
# assert exercise_5_cost3(7) == 7
# assert exercise_5_cost3(8) == 9

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: Với `n <= 6` không có khối nào lợi hơn gõ toàn chữ `A`.
- Mức 2: `dp[8]` đến từ khối x3 hoặc x4 trên `dp[4]` hoặc `dp[3]`.
- Mức 3: Kết quả: `6`, `12`; dãy phím có `A, A, A, ^A, ^C, ^V` → `3 + 3 = 6`.

**Bài 2**
- Mức 1: Ứng viên `dp[j - 1] * (i - j)`, `j` từ 2 đến `i - 2`.
- Mức 2: `dp[10]`: thử `j = 5`: `dp[4] * 5 = 20`; `j = 6`: `dp[5] * 4 = 20`; `j = 4`: `dp[3] * 6 = 18`.
- Mức 3: `dp[10] = 20`; với `dp[11]` thử `j = 7`: `dp[6] * 4 = 24`, `j = 8`: `dp[7] * 3 = 27`. Xem đáp án mục 19.

**Bài 3**
- Mức 1: `list(range(n + 1))`.
- Mức 2: `range(3, n + 1)` rồi `range(2, i - 1)`.
- Mức 3: `dp[i] = max(dp[i], dp[j - 1] * (i - j))`; `return dp[n]`.

**Bài 4**
- Mức 1: `dp[i] = max(i, dp[i - 4] * 3, dp[i - 5] * 4)` cho `i >= 7`.
- Mức 2: Vòng `for i in range(7, n + 1)`.
- Mức 3: Xem đáp án mục 19.

**Bài 5**
- Mức 1: Tổng số phím một khối là `k + 3`.
- Mức 2: Phần trước khối còn `m - k - 3` phím; cần `>= 1`.
- Mức 3: Xem `max_a_cost(n, 3)` ở mục 15.1.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [37]:
print(Solution().maxA(6))                                  # 6
print(Solution().maxA(8))                                  # 12
print(simulate(["A", "A", "A", "^A", "^C", "^V"]))         # 6

assert Solution().maxA(6) == 6
assert Solution().maxA(8) == 12
assert simulate(["A", "A", "A", "^A", "^C", "^V"]) == 6

6
12
6


### Đáp án Bài 2

`dp[10]`: ứng viên `j = 2..8`: `dp[1]*8 = 8`, `dp[2]*7 = 14`, `dp[3]*6 = 18`, `dp[4]*5 = 20`, `dp[5]*4 = 20`, `dp[6]*3 = 18`, `dp[7]*2 = 18` (vì `dp[7] = 9`). Max = **20**, tại `j = 5` (gõ 4 chữ `A`, `Ctrl-A`, `Ctrl-C`, 4 lần `Ctrl-V`: `4 * 5 = 20`) hoặc `j = 6`. `dp[11] = 27` ở `j = 8`: `dp[7] * 3`.

In [38]:
dp = list(range(12))
for i in range(3, 12):
    for j in range(2, i - 1):
        dp[i] = max(dp[i], dp[j - 1] * (i - j))
print(dp[10], dp[11])
print([(j, dp[j - 1] * (10 - j)) for j in range(2, 9)])
assert dp[10] == 20 and dp[11] == 27
assert [j for j in range(2, 9) if dp[j - 1] * (10 - j) == 20] == [5, 6]
val, keys = best_keys(10)
print(val, " ".join(keys))
assert val == 20

20 27
[(2, 8), (3, 14), (4, 18), (5, 20), (6, 20), (7, 18), (8, 18)]
20 A A A A ^A ^C ^V ^V ^V ^V


### Đáp án Bài 3

In [39]:
def exercise_3_dp_solution(n: int) -> int:
    dp = list(range(n + 1))
    for i in range(3, n + 1):
        for j in range(2, i - 1):
            dp[i] = max(dp[i], dp[j - 1] * (i - j))
    return dp[n]


assert exercise_3_dp_solution(3) == 3
assert exercise_3_dp_solution(7) == 9
assert exercise_3_dp_solution(11) == 27
assert all(exercise_3_dp_solution(n) == Solution().maxA(n) for n in range(1, 51))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_linear_solution(n: int) -> int:
    dp = list(range(n + 1))
    for i in range(7, n + 1):
        dp[i] = max(i, dp[i - 4] * 3, dp[i - 5] * 4)
    return dp[n]


assert exercise_4_linear_solution(7) == 9
assert exercise_4_linear_solution(20) == 324
assert exercise_4_linear_solution(50) == 1327104
assert all(exercise_4_linear_solution(n) == Solution().maxA(n) for n in range(1, 51))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_cost3_solution(n: int) -> int:
    f = list(range(n + 1))
    for m in range(1, n + 1):
        for k in range(1, m):
            prev = m - k - 3
            if prev >= 1:
                f[m] = max(f[m], f[prev] * (k + 1))
    return f[n]


assert exercise_5_cost3_solution(3) == 3
assert exercise_5_cost3_solution(7) == 7
assert exercise_5_cost3_solution(8) == 9
assert [exercise_5_cost3_solution(n) for n in range(1, 13)] == [max_a_cost(n, 3) for n in range(1, 13)]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def maxA(self, n: int) -> int:
        dp = list(range(n + 1))                  # phương án gõ toàn chữ A
        for i in range(3, n + 1):
            for j in range(2, i - 1):            # Ctrl-A ở lần j, rồi Ctrl-C, rồi i-j-1 lần Ctrl-V
                dp[i] = max(dp[i], dp[j - 1] * (i - j))
        return dp[-1]
```

Nhớ: một khối = `Ctrl-A, Ctrl-C` + `k` lần `Ctrl-V` → nhân `(k + 1)` với `k + 2` phím. Khối hiệu quả nhất: 4 và 5 phím.

### Checklist trước khi submit

- [ ] `dp` khởi tạo bằng `list(range(n + 1))` (gõ toàn `A` luôn là một ứng viên).
- [ ] Công thức `dp[j - 1] * (i - j)`: kiểm tra phím: `(j - 1) + 2 + (i - j - 1) = i`.
- [ ] Biên: `i` từ 3, `j` trong `range(2, i - 1)`; `n = 1, 2, 3` trả `n`.
- [ ] Đã so với bảng: `7 -> 9`, `9 -> 16`, `11 -> 27`, `50 -> 1327104`.
- [ ] Không dùng đệ quy không nhớ.
- [ ] Signature đúng: `class Solution`, `maxA(self, n: int) -> int`.